# 01. 이미지 분류 기초
Fashion-MNIST의 28×28 흑백 이미지를 10개 의류 범주로 분류합니다.
데이터 확인 → 정규화 → 분할 → 모델 생성 → 학습 → 검증 순서입니다.
최초 실행 시 데이터 다운로드를 위한 인터넷 연결이 필요합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from common.runtime import configure_runtime
from common.data import load_fashion_data

gpus = configure_runtime(seed=42)
print('GPU:', gpus)

In [ ]:
(train_x, train_y), (val_x, val_y), (test_x, test_y) = load_fashion_data()
print('훈련:', train_x.shape, train_y.shape)
print('검증:', val_x.shape, val_y.shape)
print('테스트:', test_x.shape, test_y.shape)

## 이미지와 정답 확인
훈련 48,000개, 검증 12,000개, 테스트 10,000개를 사용합니다.
원본 픽셀 0~255를 `float32`의 0~1 범위로 정규화했습니다.
`(N, H, W)`는 이 데이터의 흑백 이미지 묶음입니다. 채널을 명시하면 흑백도 `(N, H, W, 1)`이 됩니다.

In [ ]:
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
fig, axes = plt.subplots(1, 10, figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(train_x[i], cmap='gray_r')
    ax.set_title(class_names[train_y[i]], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('훈련 범주별 개수:', np.unique(train_y, return_counts=True))

## Dense 입력 모양과 분류 모델
`reshape`는 이미지 한 장을 길이 784인 벡터로 펼칩니다. 정보를 선택하는 PCA와 달리 원소를 재배열합니다.
ANN은 신경망 전체를 가리키며, 은닉층이 없는 모델에만 쓰는 용어는 아닙니다.
여기서는 은닉층 없이 `Dense(10)`을 사용한 간단한 기준 모델부터 시작합니다.

| 구성 | 의미 |
| --- | --- |
| Input(784) | 샘플 하나의 입력 모양 |
| Dense(10, softmax) | 10개 범주의 확률 출력 |
| kernel_initializer | 최초 가중치 생성 방법. 학습 중 갱신 규칙과 구별 |
| 파라미터 수 | 784×10개의 가중치 + 10개의 편향 = 7,850 |

In [ ]:
train_flat = train_x.reshape(-1, 784)
val_flat = val_x.reshape(-1, 784)
model = keras.Sequential([
    keras.Input(shape=(784,)),
    keras.layers.Dense(10, activation='softmax', kernel_initializer='uniform'),
])
model.summary()

## 손실함수와 학습
정답이 정수 0~9이므로 `sparse_categorical_crossentropy`를 사용합니다.
원-핫 정답에는 `categorical_crossentropy`, 단일 이진 출력에는 보통 `sigmoid`와 `binary_crossentropy`를 사용합니다.
ReLU는 주로 은닉층에 사용하며 그 자체가 이진 분류 확률은 아닙니다.

`epochs`는 데이터 반복 횟수, `batch_size`는 한 번의 가중치 갱신에 사용하는 샘플 수입니다.
`fit()`을 다시 호출하면 기존 가중치에서 이어 학습합니다. 새 실험은 모델부터 다시 만듭니다.
손실은 백분율이 아니며, 손실과 정확도를 함께 확인합니다.

In [ ]:
EPOCHS = 10  # 원본 30회. 수업을 재현할 때 변경합니다.
BATCH_SIZE = 32
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_flat, train_y, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    validation_data=(val_flat, val_y), verbose=2)
val_loss, val_accuracy = model.evaluate(val_flat, val_y, verbose=0)
print(f'검증 손실: {val_loss:.4f}, 정확도: {val_accuracy:.4f}')

## 복습
`model.summary()`의 학습 파라미터는 가중치와 편향입니다. 옵티마이저 상태는 학습 이후 별도로 표시될 수 있습니다.
훈련 손실과 검증 손실의 차이만으로 과적합을 단정하지 않고 곡선의 추세를 봅니다.
테스트 데이터는 튜닝에 사용하지 않으며, 최종 평가 실습은 03에서 진행합니다.